# Big Data Analytics - Book Recommendation with Implicit Feedback


Thomas Timmermans - 2365790

Milan Muroni - 2262576

In [4]:
%pip install scikit-learn implicit

In [1]:
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import implicit

In [20]:
import json

file = "../datasets/books/Books.jsonl"
likes = []

with open(file, 'r') as fp:
    for i, line in enumerate(fp):
        data = json.loads(line.strip())
        # Alleen ratings strikt boven 4.0 tellen als 'like' (x_ui = 1)
        if data.get('rating', 0) > 4.0:
            likes.append((data['user_id'], data['parent_asin']))
        
print(f"Total likes: {len(likes)}")

df = pd.DataFrame(likes, columns=['user_id', 'book_id']).drop_duplicates()




Total likes: 20391478


## data inladen en filteren

In [21]:


print(f"Total unique likes: {len(df)}")

# 1. Filter op actieve users en boeken (bijv. minimaal 15 likes per user/boek)
# Pas '15' aan (bijv. naar 10 of 20) tot je totaal tussen de 100.000 en 300.000 likes zit
min_likes = 15
for _ in range(3):
    df = df[df['user_id'].map(df['user_id'].value_counts()) >= min_likes]
    df = df[df['book_id'].map(df['book_id'].value_counts()) >= min_likes]

print(f"Total likes after filtering: {len(df)}")

# 2. Zet tekst-ID's om naar simpele getallen (0, 1, 2, ...) voor de matrix
df['user_idx'] = df['user_id'].astype('category').cat.codes
df['book_idx'] = df['book_id'].astype('category').cat.codes

# 3. Bereken de statistieken (zoals gevraagd in de opdracht)
n_users = df['user_idx'].nunique()
n_books = df['book_idx'].nunique()
n_likes = len(df)
sparsity = 1 - (n_likes / (n_users * n_books))

print(f"Users: {n_users}")
print(f"Books: {n_books}")
print(f"Positive interactions: {n_likes}")
print(f"Sparsity: {sparsity:.4%}")

# 4. Maak de sparse matrix X (voorkomt een te grote dense matrix)
X = csr_matrix((np.ones(n_likes), (df['user_idx'], df['book_idx'])), shape=(n_users, n_books))

Total unique likes: 20155617
Users: 10644
Books: 6906
Positive interactions: 243190
Sparsity: 99.6692%
